In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (2), incompatible header (7))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (4), incompatible header (14))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
openGeometry("mpc-1.geo")

#openPreProcessor()

In [3]:
mat1 = Material("body")
mat2 = Material("remote")

U = Field([mat1, mat2], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
Φ = Field([mat1, mat2], type=:ScalarField, dim=2, fieldName=:φ, rhsName=:m)

Problem("mpc-1", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)

In [4]:
Ku = ∫(SymGrad(U) ⋅ [2 1 0; 1 2 0; 0 0 1] ⋅ SymGrad(U), Ω="body")
Ku[:,:]

10×10 SparseArrays.SparseMatrixCSC{Float64, Int64} with 62 stored entries:
  1.0           0.5          -0.5          …  -1.38778e-17   ⋅    ⋅ 
  0.5           1.0          -3.46945e-18     -0.5           ⋅    ⋅ 
 -0.5          -3.46945e-18   1.0              0.5           ⋅    ⋅ 
  1.04083e-17   5.55112e-17  -0.5             -0.5           ⋅    ⋅ 
 -0.5          -0.5           5.55112e-17       ⋅            ⋅    ⋅ 
 -0.5          -0.5           1.38778e-17  …   5.55112e-17   ⋅    ⋅ 
  5.55112e-17   1.38778e-17  -0.5             -0.5           ⋅    ⋅ 
 -1.38778e-17  -0.5           0.5              1.0           ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 

In [5]:
mpc_u = MPC(master="remote", slave="right", field=U, ux=true, uy=true)

mpc_φ = MPC(master="remote", slave="right", field=Φ)

bc2 = BoundaryCondition("remote", field=U, ux=0, uy=0)

BoundaryCondition("remote", Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false), Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [6]:
R = rigidRotationMap(mpc_u, mpc_φ)
R[:,:]

10×5 SparseArrays.SparseMatrixCSC{Float64, Int64} with 2 stored entries:
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅   0.5    ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅   -0.5   ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 

In [7]:
Kuφ = Ku * R
Kuφ[:,:]

10×5 SparseArrays.SparseMatrixCSC{Float64, Int64} with 15 stored entries:
  ⋅   -0.25          0.25          ⋅    ⋅ 
  ⋅   -1.73472e-18   0.25          ⋅    ⋅ 
  ⋅    0.5          -2.77556e-17   ⋅    ⋅ 
  ⋅   -0.25          6.93889e-18   ⋅    ⋅ 
  ⋅    2.77556e-17  -0.5           ⋅    ⋅ 
  ⋅    6.93889e-18  -0.25          ⋅    ⋅ 
  ⋅   -0.25          0.25          ⋅    ⋅ 
  ⋅    0.25           ⋅            ⋅    ⋅ 
  ⋅     ⋅             ⋅            ⋅    ⋅ 
  ⋅     ⋅             ⋅            ⋅    ⋅ 

In [8]:
Kφ = R' * Ku * R
Kφ[:,:]

5×5 SparseArrays.SparseMatrixCSC{Float64, Int64} with 4 stored entries:
  ⋅     ⋅             ⋅            ⋅    ⋅ 
  ⋅    0.25         -1.38778e-17   ⋅    ⋅ 
  ⋅   -1.38778e-17   0.25          ⋅    ⋅ 
  ⋅     ⋅             ⋅            ⋅    ⋅ 
  ⋅     ⋅             ⋅            ⋅    ⋅ 

In [9]:
K = SystemMatrix([Ku Kuφ; Kuφ' Kφ])
K[:,:]

15×15 SparseArrays.SparseMatrixCSC{Float64, Int64} with 96 stored entries:
  1.0           0.5          -0.5          …   0.25          ⋅    ⋅ 
  0.5           1.0          -3.46945e-18      0.25          ⋅    ⋅ 
 -0.5          -3.46945e-18   1.0             -2.77556e-17   ⋅    ⋅ 
  1.04083e-17   5.55112e-17  -0.5              6.93889e-18   ⋅    ⋅ 
 -0.5          -0.5           5.55112e-17     -0.5           ⋅    ⋅ 
 -0.5          -0.5           1.38778e-17  …  -0.25          ⋅    ⋅ 
  5.55112e-17   1.38778e-17  -0.5              0.25          ⋅    ⋅ 
 -1.38778e-17  -0.5           0.5               ⋅            ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 
   ⋅             ⋅             ⋅           …    ⋅            ⋅    ⋅ 
 -0.25         -1.73472e-18   0.5             -1.38778e-17   ⋅    ⋅ 
  0.25          0.25         -2.77556e-17      0.25          ⋅    ⋅ 
   ⋅             ⋅          

In [10]:
bc = BoundaryCondition("left", field=U, ux=0, uy=0)

fu = ∫(U ⋅ [0, 0], Γ="remote")
fφ = ∫(Φ ⋅ 1, Γ="remote")

nodal ScalarField
[0.0; 0.0; … ; 0.0; 1.0;;]

In [11]:
DoFs(fu)

10×1 Matrix{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [12]:
DoFs(fφ)

5×1 Matrix{Float64}:
 0.0
 0.0
 0.0
 0.0
 1.0

In [13]:
F = SystemVector([fu, fφ])

SystemVector([0.0; 0.0; … ; 0.0; 1.0;;], nothing, Problem[Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false), Problem("mpc-1", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)], [0, 10])

In [14]:
u, φ = solveField(K, F, support=[bc], mpc=[mpc_u, mpc_φ])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.0; 2.0;;], [0.0], Int64[], 1, :v2D, Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)), ScalarField(Matrix{Float64}[], [0.0; 4.0; … ; 0.0; 4.0;;], [0.0], Int64[], 1, :scalar, Problem("mpc-1", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0

In [15]:
showDoFResults(u, name="u", visible=true)

0

In [16]:
DoFs(φ)

5×1 Matrix{Float64}:
 0.0
 4.0
 4.0
 0.0
 4.0

In [17]:
u2 = u + R * φ
DoFs(u2)

10×1 Matrix{Float64}:
  0.0
  0.0
  2.0
  2.0
 -2.0
  2.0
  0.0
  0.0
  0.0
  2.0

In [18]:
showDoFResults(u2, name="u2")

1

In [19]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
